# Bagging (Bootstrap Aggregating) from scratch
Sections follow the lesson:
1. Decision tree helper (weighted, with optional random feature subsets)
2. Bootstrap sampling and the 63.2% rule
3. Aggregation: majority / soft vote and averaging
4. `BaggingClassifier` with OOB evaluation
5. Why bagging reduces variance: `rho*s^2 + (1-rho)*s^2/M`
6. Bagging -> Random Forest (`max_features = sqrt(p)`), effect of hyperparameters
7. Bagging does not fix bias (stable learners)

In [ ]:
import os
import warnings

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.simplefilter("ignore")
os.makedirs("plots", exist_ok=True)
rng = np.random.default_rng(42)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 110})

## 1. A small decision tree (CART, variance-reduction splits)
For 0/1 labels the variance of the labels equals Gini/2, so one regression-style tree
handles both regression (leaf = mean) and classification (leaf = P(y=1)).
Leaf values are stored so `predict` returns a probability for classification.

In [ ]:
class DecisionTree:
    """Greedy CART tree. Leaf value = (weighted) mean of targets.

    max_features: None -> consider all features at every split;
                  int  -> random subset of that size at every split (Random Forest).
    """

    def __init__(self, max_depth=None, min_samples_leaf=1, max_features=None, rng=None):
        self.max_depth = np.inf if max_depth is None else max_depth
        self.msl = min_samples_leaf
        self.max_features = max_features
        self.rng = rng if rng is not None else np.random.default_rng(0)

    def fit(self, X, y, w=None):
        w = np.ones(len(y)) if w is None else np.asarray(w, float)
        self.feat, self.thr, self.left, self.right, self.value = [], [], [], [], []
        self._grow(X, y.astype(float), w, 0)
        for name in ("feat", "thr", "left", "right", "value"):
            setattr(self, name, np.array(getattr(self, name)))
        return self

    def _new_node(self):
        for lst, v in ((self.feat, -1), (self.thr, 0.0), (self.left, -1), (self.right, -1), (self.value, 0.0)):
            lst.append(v)
        return len(self.feat) - 1

    def _grow(self, X, y, w, depth):
        node = self._new_node()
        sw = w.sum()
        self.value[node] = (w * y).sum() / sw if sw > 0 else y.mean()
        if depth >= self.max_depth or len(y) < 2 * self.msl or np.all(y == y[0]):
            return node
        j, t = self._best_split(X, y, w)
        if j is None:
            return node
        mask = X[:, j] <= t
        self.feat[node], self.thr[node] = j, t
        self.left[node] = self._grow(X[mask], y[mask], w[mask], depth + 1)
        self.right[node] = self._grow(X[~mask], y[~mask], w[~mask], depth + 1)
        return node

    def _best_split(self, X, y, w):
        n, d = X.shape
        feats = np.arange(d)
        if self.max_features is not None and self.max_features < d:
            feats = self.rng.choice(d, self.max_features, replace=False)
        Tw, Ty = w.sum(), (w * y).sum()
        best_gain, best = 1e-12, (None, None)
        parent = Ty**2 / Tw
        for j in feats:
            o = np.argsort(X[:, j], kind="stable")
            xs = X[o, j]
            cw, cy = np.cumsum(w[o])[:-1], np.cumsum((w * y)[o])[:-1]
            valid = xs[:-1] < xs[1:]
            valid[: self.msl - 1] = False
            valid[n - self.msl:] = False
            valid &= (cw > 1e-12) & (Tw - cw > 1e-12)
            if not valid.any():
                continue
            gain = np.where(valid, cy**2 / np.maximum(cw, 1e-12) + (Ty - cy) ** 2 / np.maximum(Tw - cw, 1e-12) - parent, -np.inf)
            i = int(np.argmax(gain))
            if gain[i] > best_gain:
                best_gain, best = gain[i], (j, 0.5 * (xs[i] + xs[i + 1]))
        return best

    def apply(self, X):
        """Leaf id of every row."""
        node = np.zeros(len(X), dtype=int)
        active = self.left[node] != -1
        while active.any():
            idx = np.where(active)[0]
            nd = node[idx]
            go_left = X[idx, self.feat[nd]] <= self.thr[nd]
            node[idx] = np.where(go_left, self.left[nd], self.right[nd])
            active = self.left[node] != -1
        return node

    def predict(self, X):
        return self.value[self.apply(X)]


# synthetic binary classification data (non-linear, 20 features, 5 informative, 5% label noise)
def make_data(n, d=20, rng=rng, flip=0.05):
    X = rng.normal(size=(n, d))
    score = X[:, 0] + X[:, 1] * X[:, 2] + 0.8 * X[:, 3] ** 2 - 0.8 + 0.7 * X[:, 4] * (X[:, 0] > 0)
    y = (score + 0.3 * rng.normal(size=n) > 0).astype(int)
    y = np.where(rng.random(n) < flip, 1 - y, y)
    return X, y


Xtr, ytr = make_data(700)
Xte, yte = make_data(1500)
acc = lambda p, y: float(np.mean((p >= 0.5).astype(int) == y))
print("train", Xtr.shape, "test", Xte.shape, "positive rate", ytr.mean().round(2))

single = DecisionTree(rng=np.random.default_rng(1)).fit(Xtr, ytr)
print("single deep tree: train acc %.3f | test acc %.3f" % (acc(single.predict(Xtr), ytr), acc(single.predict(Xte), yte)))

## 2. Bootstrap sampling
Draw N rows *with replacement*. Some rows repeat, others are never drawn (the OOB rows).
`P(row never picked) = (1-1/N)^N -> e^-1 = 0.368`, so ~63.2% unique rows per sample.

In [ ]:
def bootstrap_indices(n, rng):
    """Indices of one bootstrap sample of size n, plus the boolean OOB mask."""
    idx = rng.integers(0, n, n)
    oob = np.ones(n, dtype=bool)
    oob[idx] = False
    return idx, oob


# the doc's 5-row example
letters = np.array(list("ABCDE"))
for k in range(2):
    idx, oob = bootstrap_indices(5, np.random.default_rng(k + 3))
    print(f"bootstrap {k+1}:", "".join(letters[idx]), "| OOB rows:", "".join(letters[oob]) or "-")

# empirical vs theory
sizes = [5, 10, 20, 50, 100, 500, 2000]
emp = [np.mean([1 - bootstrap_indices(n, rng)[1].mean() for _ in range(300)]) for n in sizes]
theory = [1 - (1 - 1 / n) ** n for n in sizes]
for n, e, t in zip(sizes, emp, theory):
    print(f"N={n:5d}  unique fraction empirical={e:.3f} theory={t:.3f}  (limit {1-np.exp(-1):.3f})")

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogx(sizes, emp, "o", label="empirical (300 draws)")
ax.semilogx(sizes, theory, "-", label="1-(1-1/N)^N")
ax.axhline(1 - np.exp(-1), color="k", ls="--", label="1-1/e = 0.632")
ax.set(xlabel="dataset size N", ylabel="fraction of unique rows in a bootstrap", title="The 63.2% rule")
ax.legend(); fig.tight_layout(); fig.savefig("plots/01_bootstrap_632.png"); plt.close(fig)

## 3. Aggregation
* Classification: majority vote (hard) or mean of probabilities (soft).
* Regression: plain average.

In [ ]:
votes = np.array(["Cat", "Dog", "Cat", "Cat", "Dog"])
vals, counts = np.unique(votes, return_counts=True)
print("votes:", dict(zip(vals, counts)), "->", vals[counts.argmax()])
print("average of [110,120,115,118] =", np.mean([110, 120, 115, 118]))

## 4. Bagged trees with out-of-bag evaluation
Each tree is trained on its own bootstrap sample. For every training row we aggregate
**only the trees that never saw it** -> a free validation estimate.

In [ ]:
class Bagging:
    """Bagging of DecisionTrees (task='clf' uses 0/1 labels, 'reg' averages).

    max_features=None -> plain bagging;  max_features='sqrt' / int -> Random Forest.
    """

    def __init__(self, n_estimators=100, max_depth=None, min_samples_leaf=1, max_features=None,
                 max_samples=1.0, bootstrap=True, seed=0):
        self.M, self.max_depth, self.msl = n_estimators, max_depth, min_samples_leaf
        self.max_features, self.max_samples, self.bootstrap, self.seed = max_features, max_samples, bootstrap, seed

    def fit(self, X, y):
        n, d = X.shape
        rng = np.random.default_rng(self.seed)
        mf = int(np.sqrt(d)) if self.max_features == "sqrt" else self.max_features
        m = int(self.max_samples * n)
        self.trees, self.oob_masks = [], []
        for _ in range(self.M):
            if self.bootstrap:
                idx = rng.integers(0, n, m)
                oob = np.ones(n, bool); oob[idx] = False
            else:
                idx, oob = np.arange(n), np.zeros(n, bool)
            t = DecisionTree(self.max_depth, self.msl, mf, np.random.default_rng(rng.integers(1 << 31)))
            self.trees.append(t.fit(X[idx], y[idx]))
            self.oob_masks.append(oob)
        self.X_, self.y_ = X, y
        return self

    def tree_preds(self, X):
        return np.array([t.predict(X) for t in self.trees])          # (M, n)

    def predict_proba(self, X, n_trees=None):
        """Soft vote: mean of tree leaf probabilities."""
        return self.tree_preds(X)[: n_trees or self.M].mean(axis=0)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

    def predict_hard(self, X):
        """Hard majority vote."""
        return ((self.tree_preds(X) >= 0.5).mean(axis=0) >= 0.5).astype(int)

    def oob_predict_proba(self):
        """Aggregate over trees whose bootstrap sample did NOT contain the row."""
        P = self.tree_preds(self.X_)
        Mk = np.array(self.oob_masks)
        cnt = Mk.sum(axis=0)
        with np.errstate(invalid="ignore", divide="ignore"):
            p = (P * Mk).sum(axis=0) / cnt
        return p, cnt > 0

    def oob_score(self):
        p, ok = self.oob_predict_proba()
        return float(np.mean((p[ok] >= 0.5).astype(int) == self.y_[ok]))


bag = Bagging(n_estimators=100, seed=1).fit(Xtr, ytr)
print("Bagging  OOB score: %.3f | test acc (soft): %.3f | test acc (hard): %.3f | train acc: %.3f" %
      (bag.oob_score(), acc(bag.predict_proba(Xte), yte), float(np.mean(bag.predict_hard(Xte) == yte)), acc(bag.predict_proba(Xtr), ytr)))
p_oob, ok = bag.oob_predict_proba()
print("avg #OOB trees per row: %.1f of %d (expected ~%.1f)" % (np.mean([m for m in np.array(bag.oob_masks).sum(0)]), bag.M, 0.368 * bag.M))

## 5. Random Forest = bagging + random feature subsets
At every split only `sqrt(p)` random features are considered, which decorrelates the trees.

In [ ]:
rf = Bagging(n_estimators=100, max_features="sqrt", seed=2).fit(Xtr, ytr)
print("RF       OOB score: %.3f | test acc: %.3f" % (rf.oob_score(), acc(rf.predict_proba(Xte), yte)))
print("Single tree test acc: %.3f" % acc(single.predict(Xte), yte))

# accuracy vs number of trees (single fit, cumulative average)
ks = [1, 2, 3, 5, 8, 12, 20, 30, 50, 75, 100]
def curve(model):
    P = model.tree_preds(Xte)
    return [acc(P[:k].mean(0), yte) for k in ks]
c_bag, c_rf = curve(bag), curve(rf)

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(ks, c_bag, "o-", label="Bagging (all features)")
ax.plot(ks, c_rf, "s-", label="Random Forest (sqrt features)")
ax.axhline(acc(single.predict(Xte), yte), color="gray", ls="--", label="single deep tree")
ax.axhline(bag.oob_score(), color="C0", ls=":", label="bagging OOB score")
ax.set(xlabel="number of trees M", ylabel="test accuracy", title="More trees: diminishing returns, no overfitting")
ax.legend(); fig.tight_layout(); fig.savefig("plots/02_accuracy_vs_trees.png"); plt.close(fig)

## 6. OOB estimate vs held-out test
OOB error as a function of ensemble size (only rows with at least one OOB tree are scored).

In [ ]:
def oob_curve(model, ks):
    P, Mk = model.tree_preds(model.X_), np.array(model.oob_masks)
    out = []
    for k in ks:
        cnt = Mk[:k].sum(0); ok = cnt > 0
        p = (P[:k] * Mk[:k]).sum(0)[ok] / cnt[ok]
        out.append(float(np.mean((p >= 0.5).astype(int) == model.y_[ok])))
    return out

ks2 = [3, 5, 8, 12, 20, 30, 50, 75, 100]
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(ks2, oob_curve(bag, ks2), "o-", label="OOB accuracy (bagging)")
ax.plot(ks, c_bag, "s--", label="test accuracy (bagging)")
ax.set(xlabel="number of trees M", ylabel="accuracy", title="OOB tracks test accuracy without a validation split")
ax.legend(); fig.tight_layout(); fig.savefig("plots/03_oob_vs_test.png"); plt.close(fig)
print("final OOB %.3f vs test %.3f" % (oob_curve(bag, [100])[0], c_bag[-1]))

## 7. The math: `Var(ensemble) = rho*s^2 + (1-rho)*s^2/M`
Empirically measure, over many training sets, the variance of one tree vs bagged trees at fixed test
points, and the average correlation `rho` between trees' predictions. Random feature subsets lower rho.

In [ ]:
def make_reg(n, rng):
    X = rng.uniform(-1, 1, size=(n, 6))
    y = np.sin(3 * X[:, 0]) + X[:, 1] ** 2 + X[:, 2] * X[:, 3] + 0.3 * rng.normal(size=n)
    return X, y


Xg = np.random.default_rng(99).uniform(-1, 1, size=(80, 6))         # fixed test points
Ms = [1, 3, 10, 30]
n_ds = 25
results = {}
for name, mf in (("bagging", None), ("random forest", 2)):
    preds = np.zeros((n_ds, max(Ms), len(Xg)))
    for d in range(n_ds):
        r = np.random.default_rng(1000 + d)
        X, y = make_reg(150, r)
        for m in range(max(Ms)):
            idx = r.integers(0, 150, 150)
            t = DecisionTree(rng=np.random.default_rng(r.integers(1 << 31)), max_features=mf).fit(X[idx], y[idx])
            preds[d, m] = t.predict(Xg)
    var_M = [preds[:, :M].mean(1).var(axis=0).mean() for M in Ms]        # variance over training sets
    s2 = preds[:, 0].var(axis=0).mean()                                  # single-tree variance
    # rho: variance-weighted average correlation between trees (same dataset, different bootstraps)
    dev = preds - preds.mean(axis=1, keepdims=True)                      # within-dataset deviation
    rho = 1 - (dev[:, :30].var(axis=1, ddof=1).mean() / s2) if False else None
    var_M30 = var_M[-1]
    rho = (var_M30 - s2 / 30) / (s2 - s2 / 30)                           # solve formula for rho at M=30
    results[name] = (var_M, s2, rho)
    print(f"{name:14s} var by M {dict(zip(Ms, np.round(var_M, 4)))}  single-tree var={s2:.4f}  implied rho={rho:.2f}")

fig, ax = plt.subplots(figsize=(6.5, 4))
for name, (var_M, s2, rho) in results.items():
    line, = ax.plot(Ms, var_M, "o-", label=f"{name} (empirical, rho~{rho:.2f})")
    ax.plot(Ms, [rho * s2 + (1 - rho) * s2 / M for M in Ms], "--", color=line.get_color(), alpha=0.6)
ax.set(xscale="log", xlabel="trees M", ylabel="prediction variance over training sets",
       title="Variance floor rho*s^2 (dashed = formula)")
ax.legend(); fig.tight_layout(); fig.savefig("plots/04_variance_floor.png"); plt.close(fig)

## 8. Hyperparameters: `max_samples` and `max_depth`
* smaller bootstrap samples -> more diversity but more bias;
* bagging works best with deep low-bias trees; on stumps (high bias) it is far less useful.

In [ ]:
print("max_samples effect (40 trees):")
ms_list = [0.1, 0.25, 0.5, 1.0]
ms_acc = []
for ms in ms_list:
    b = Bagging(n_estimators=40, max_samples=ms, seed=5).fit(Xtr, ytr)
    ms_acc.append(acc(b.predict_proba(Xte), yte)); print(f"  max_samples={ms:4.2f} test acc={ms_acc[-1]:.3f}")

depths = [1, 2, 4, 8, None]
d_single, d_bag = [], []
for dep in depths:
    t = DecisionTree(max_depth=dep, rng=np.random.default_rng(0)).fit(Xtr, ytr)
    b = Bagging(n_estimators=40, max_depth=dep, seed=6).fit(Xtr, ytr)
    d_single.append(acc(t.predict(Xte), yte)); d_bag.append(acc(b.predict_proba(Xte), yte))
    print(f"  max_depth={str(dep):4s} single={d_single[-1]:.3f} bagged={d_bag[-1]:.3f} gain={d_bag[-1]-d_single[-1]:+.3f}")

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].plot(ms_list, ms_acc, "o-"); axs[0].set(xlabel="max_samples (fraction)", ylabel="test accuracy", title="Smaller bootstraps raise bias")
xl = [str(d) for d in depths]
axs[1].plot(xl, d_single, "o-", label="single tree"); axs[1].plot(xl, d_bag, "s-", label="bagged (40)")
axs[1].set(xlabel="max_depth", ylabel="test accuracy", title="Bagging helps most for deep trees"); axs[1].legend()
fig.tight_layout(); fig.savefig("plots/05_hyperparameters.png"); plt.close(fig)

## 9. Optional sanity check against scikit-learn (dataset-agnostic, comparison only)

In [ ]:
try:
    from sklearn.ensemble import BaggingClassifier
    from sklearn.tree import DecisionTreeClassifier
    sk = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, oob_score=True, random_state=0).fit(Xtr, ytr)
    print("sklearn Bagging: OOB %.3f  test %.3f   (ours: OOB %.3f  test %.3f)" %
          (sk.oob_score_, sk.score(Xte, yte), bag.oob_score(), acc(bag.predict_proba(Xte), yte)))
except Exception as e:  # sklearn optional
    print("sklearn not available, skipping:", type(e).__name__)